# 04 – Error taxonomy: *how* is the model wrong?

`02`/`03` found **where** the model fails. Here we classify each of the wrong test outputs by **what kind of mistake**
it is, then check whether the hotspots fail in different ways.

Every target and (almost) every prediction is *linear MBA*: a sum of `coefficient * (bitwise term)` plus a constant,
e.g. `-2*(x|y)+1`. That lets us compare prediction and target term by term:

| category | meaning | example (target → prediction) |
|---|---|---|
| swapped variables | correct up to renaming variables | `x&~y` → `~x&y` |
| sign flip | prediction = −target | `x^y` → `-(x^y)` |
| off by a constant | only the constant is wrong (incl. dropped/added constant) | `-(~x)-1` → `-(~x)` |
| wrong coefficient | same bitwise terms, different coefficients | `-2*(x|~y)` → `-(x|~y)` |
| wrong bitwise function | same number of terms and coefficients, but a different bitwise function | `x&y` → `x|~y` |
| missing / extra term | the prediction has fewer / more terms | `-(x&y)-(x|y)+1` → `-(x|y)+1` |
| several changes | anything else | |

The first three are checked **semantically** (on the truth-table signature), the rest **structurally** (on the terms as
written). Caveat: a linear-MBA function can be written with different terms (`x|y` = `x+y-(x&y)`), so "missing term"
can occasionally mean "wrote it with different building blocks".

In [ ]:
import ast
import itertools
import re
import sys
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "predictions_test.csv").exists())
sys.path.insert(0, str(ROOT / "src"))
import neureduce as nr

FIG = Path.cwd() / "figures"
FIG.mkdir(exist_ok=True)

df = pd.read_csv(ROOT / "data" / "predictions_test.csv")
df["error"] = ~df["correct"]
wrong = df[df.verdict == "wrong"].copy()
print(f"{len(df):,} test examples, {len(wrong):,} wrong (+ {(df.verdict == 'invalid').sum()} unparseable outputs)")

## 1. Classify every wrong output

In [ ]:
VARS = lambda *exprs: sorted(set(re.findall(r"[a-z]", "".join(exprs))))


def signature(tree, names, rename=None):
    """Values on all {0,-1} corners (exact identity for linear MBA). `rename` permutes variables."""
    rename = rename or {v: v for v in names}
    out = []
    for bits in itertools.product((0, -1), repeat=len(names)):
        env = dict(zip(names, bits))
        out.append(-nr.evaluate(tree, {v: env[rename[v]] for v in names}))
    return np.array(out)


def truth_table(node, names):
    """1-bit truth table of a pure bitwise subexpression."""
    return tuple(nr.evaluate(node, dict(zip(names, bits))) & 1
                 for bits in itertools.product((0, 1), repeat=len(names)))


def linear_terms(tree, names):
    """Linear MBA -> ({truth table: coefficient}, constant). Raises ValueError if not linear."""
    terms, const = Counter(), 0

    def walk(n, sign):
        nonlocal const
        if isinstance(n, ast.BinOp) and isinstance(n.op, (ast.Add, ast.Sub)):
            walk(n.left, sign)
            walk(n.right, sign if isinstance(n.op, ast.Add) else -sign)
        elif nr._const(n):
            const += sign * nr.evaluate(n, {})
        elif isinstance(n, ast.UnaryOp) and isinstance(n.op, (ast.USub, ast.UAdd)):
            walk(n.operand, -sign if isinstance(n.op, ast.USub) else sign)
        elif nr._bitwise(n):
            terms[truth_table(n, names)] += sign
        elif isinstance(n, ast.BinOp) and isinstance(n.op, ast.Mult) and nr._const(n.left):
            walk(n.right, sign * nr.evaluate(n.left, {}))
        elif isinstance(n, ast.BinOp) and isinstance(n.op, ast.Mult) and nr._const(n.right):
            walk(n.left, sign * nr.evaluate(n.right, {}))
        else:
            raise ValueError("not linear MBA")

    walk(tree, 1)
    return {t: c for t, c in terms.items() if c}, const


def classify(src, trg, pred):
    names = VARS(src, trg, pred)
    t_tree, p_tree = nr.parse(trg), nr.parse(pred)
    st, sp = signature(t_tree, names), signature(p_tree, names)
    wrong_share = float(np.mean(st != sp))  # share of bit patterns on which the answer is wrong
    for perm in itertools.permutations(names):
        if perm != tuple(names) and np.array_equal(signature(p_tree, names, dict(zip(names, perm))), st):
            return "swapped variables", wrong_share
    if np.array_equal(sp, -st):
        return "sign flip", wrong_share
    diff = sp - st
    if np.all(diff == diff[0]):
        return "off by a constant", wrong_share
    try:
        P, _ = linear_terms(p_tree, names)
        T, _ = linear_terms(t_tree, names)
    except ValueError:
        return "non-linear output", wrong_share
    if set(P) == set(T):
        return "wrong coefficient", wrong_share
    if len(P) == len(T):
        same_coefs = sorted(P.values()) == sorted(T.values())
        return ("wrong bitwise function" if same_coefs else "several changes"), wrong_share
    return ("missing term" if len(P) < len(T) else "extra term"), wrong_share


res = [classify(s, t, p) for s, t, p in zip(wrong.src, wrong.trg, wrong.pred)]
wrong["category"] = [c for c, _ in res]
wrong["wrong_share"] = np.array([w for _, w in res], dtype=float)
CATS = ["swapped variables", "sign flip", "off by a constant", "wrong coefficient",
        "wrong bitwise function", "missing term", "extra term", "several changes", "non-linear output"]
CATS = [c for c in CATS if c in set(wrong.category)]
counts = wrong.category.value_counts().reindex(CATS)
display(pd.DataFrame({"n": counts, "share": (counts / len(wrong)).round(3)}))

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 0.4 * len(CATS) + 1))
ax.barh(CATS, counts.values, color="tab:red")
for y, n in enumerate(counts.values):
    ax.text(n + 3, y, f"{n}  ({n / len(wrong):.0%})", va="center", fontsize=8)
ax.invert_yaxis()
ax.set_xlabel("wrong test outputs")
ax.set_title(f"What kind of mistake? ({len(wrong):,} wrong outputs)")
ax.set_xlim(0, counts.max() * 1.25)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(FIG / "taxonomy_categories.png", dpi=200)

### Examples per category

Always read some – check that the rules put examples where you'd put them by hand.

In [ ]:
pd.set_option("display.max_colwidth", 60)
(wrong.sample(frac=1, random_state=0).groupby("category").head(3)
      .assign(order=lambda d: d.category.map(CATS.index)).sort_values("order")
      [["category", "src", "trg", "pred"]])

## 2. How wrong? Share of bit patterns with a wrong answer

For each wrong output: on what share of the possible bit patterns of the variables (the {0,-1} corners) does the
prediction give a different value than the target? Near 0 = "almost right", 1 = wrong everywhere.

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 3.2))
bins = np.linspace(0, 1, 17)
ax.hist(wrong.wrong_share, bins=bins, color="tab:red", alpha=0.85)
ax.set_xlabel("share of bit patterns where the prediction is wrong")
ax.set_ylabel("wrong outputs")
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(FIG / "taxonomy_wrong_share.png", dpi=200)
wrong.groupby("category").wrong_share.describe()[["count", "mean", "50%"]].reindex(CATS).round(2)

## 3. Short bitwise answers: which function does the model confuse with which?

The single largest error block in `02` was single-term targets of 3–5 characters (`x&y`, `~x&y`, `x|~y`, …).
Rows = the 12 most frequent such targets; columns = what the model answered (row-normalized: each row sums to 100%).

In [ ]:
short = df[(df.trg_n_terms == 1) & (df.trg_len <= 5)].copy()
top_targets = short.trg.value_counts().head(12).index
s = short[short.trg.isin(top_targets)].copy()
top_wrong = s.loc[s.error, "pred"].value_counts().head(10).index
s["answer"] = np.where(~s.error, "(correct)", np.where(s.pred.isin(top_wrong), s.pred, "(other wrong)"))
cols = ["(correct)"] + list(top_wrong) + ["(other wrong)"]
conf = pd.crosstab(s.trg, s.answer).reindex(index=top_targets, columns=cols, fill_value=0)
share = conf.div(conf.sum(axis=1), axis=0)

fig, ax = plt.subplots(figsize=(1.0 * len(cols) + 2, 0.5 * len(top_targets) + 1.8))
im = ax.imshow(share.where(share > 0), cmap="Reds", vmin=0, vmax=0.4, aspect="auto")
for i in range(share.shape[0]):
    for j in range(share.shape[1]):
        if conf.iat[i, j]:
            ax.text(j, i, f"{share.iat[i, j]:.0%}\n{conf.iat[i, j]}", ha="center", va="center", fontsize=7,
                    color="white" if share.iat[i, j] > 0.25 else "black")
ax.set_xticks(range(len(cols)), cols, rotation=45, ha="right", fontsize=8)
ax.set_yticks(range(len(top_targets)), [f"{t}  (n={conf.loc[t].sum()})" for t in top_targets], fontsize=8)
ax.set_xlabel("model output")
ax.set_ylabel("correct answer")
ax.set_title("Short bitwise targets: what the model answers (row share, count; colour capped at 40%)", fontsize=9)
fig.colorbar(im, ax=ax, fraction=0.03)
fig.tight_layout()
fig.savefig(FIG / "taxonomy_bitwise_confusion.png", dpi=200)

In [ ]:
# truth tables make the confusions readable: rows (x,y) = 00, 01, 10, 11
def tt2(expr):
    return "".join(str(nr.evaluate(nr.parse(expr), {"x": a, "y": b}) & 1) for a in (0, 1) for b in (0, 1)) \
        if set(re.findall(r"[a-z]", expr)) <= {"x", "y"} else "-"

pairs = (s[s.error].groupby(["trg", "pred"]).size().sort_values(ascending=False).head(12)
         .rename("n").reset_index())
pairs["trg_truth_table"] = pairs.trg.map(tt2)
pairs["pred_truth_table"] = pairs.pred.map(tt2)
pairs

## 4. Do the hotspots fail in different ways?

Wrong outputs grouped by the hotspots from `02`/`03` (groups are exclusive, assigned in this order).
If a hotspot has its own typical mistake, that's a strong lead for Phase II.

In [ ]:
def group(r):
    if r.src_n_vars == 3:
        return "3-variable input"
    if r.trg_n_terms == 1 and r.trg_len <= 5:
        return "short bitwise target"
    if r.trg_len >= 15:
        return "long target (>= 15)"
    if r.src_n_terms <= 3:
        return "few input terms (<= 3)"
    return "everything else"

GROUPS = ["3-variable input", "short bitwise target", "long target (>= 15)", "few input terms (<= 3)", "everything else"]
wrong["group"] = wrong.apply(group, axis=1)
tab = pd.crosstab(wrong.group, wrong.category).reindex(index=GROUPS, columns=CATS, fill_value=0)
tab_share = tab.div(tab.sum(axis=1), axis=0)

fig, ax = plt.subplots(figsize=(9, 3.6))
left = np.zeros(len(GROUPS))
colors = plt.get_cmap("tab10").colors
for k, cat in enumerate(CATS):
    ax.barh(GROUPS, tab_share[cat], left=left, color=colors[k % 10], label=cat)
    for y, (l, w) in enumerate(zip(left, tab_share[cat])):
        if w >= 0.07:
            ax.text(l + w / 2, y, f"{w:.0%}", ha="center", va="center", fontsize=7, color="white")
    left += tab_share[cat].to_numpy()
ax.set_yticks(range(len(GROUPS)), [f"{g}  (n={tab.loc[g].sum()})" for g in GROUPS], fontsize=8)
ax.invert_yaxis()
ax.set_xlim(0, 1)
ax.set_xlabel("share of the group's wrong outputs")
ax.legend(fontsize=7, bbox_to_anchor=(1.01, 1), loc="upper left")
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(FIG / "taxonomy_by_hotspot.png", dpi=200)
tab

## 5. Does the model know when it is wrong?

`min_tok_prob` = probability of the least confident character. If wrong outputs have clearly lower values, confidence
could flag likely mistakes; if some categories are made *confidently*, those are the most dangerous errors.

In [ ]:
df["category"] = "correct"
df.loc[wrong.index, "category"] = wrong.category
order = ["correct"] + CATS
data = [df.loc[df.category == c, "min_tok_prob"] for c in order]
fig, ax = plt.subplots(figsize=(8, 3.6))
ax.boxplot(data, orientation="horizontal", showfliers=False, widths=0.6)
ax.set_yticks(range(1, len(order) + 1), [f"{c}  (n={len(d)})" for c, d in zip(order, data)], fontsize=8)
ax.invert_yaxis()
ax.set_xlabel("min_tok_prob (confidence at the least certain character)")
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(FIG / "taxonomy_confidence.png", dpi=200)

for thr in (0.5, 0.7, 0.9):
    flagged = df.min_tok_prob < thr
    print(f"min_tok_prob < {thr}: flags {flagged.mean():.1%} of outputs, catches "
          f"{df.error[flagged].sum() / df.error.sum():.0%} of errors (precision {df.error[flagged].mean():.0%})")

## Writing it up

> *"Most wrong outputs are [category] (X%), e.g. … . Short bitwise answers are mostly confused with …
> 3-variable inputs instead fail by … . The model is [/ is not] less confident on its mistakes."*

## Save the categories for `06`

`06_projections` colours the projections by mistake type; it reads this file.

In [ ]:
out = Path.cwd() / "taxonomy.csv"
wrong[["row", "category", "wrong_share"]].to_csv(out, index=False)
print(f"wrote {out.name} ({len(wrong)} rows)")